# PERA-SAM Cloud Trainer (Kaggle Edition)

> **Run this notebook on Kaggle** with the MIMII datasets pre-mounted as Kaggle datasets.
>
> **Before running:**
> 1. Add the Kaggle MIMII datasets to this notebook via **Add Data** (top-right):
>    - Search `MIMII pump` → add `daisukelab/mimii-dataset` (or search `mimii pump valve slider`)
>    - The dataset will be mounted at `/kaggle/input/mimii-dataset/` automatically — **no download needed**.
> 2. Run all cells in order.
> 3. Download `pera_sam_models.zip` from the **Output** panel (right side).

**Output:** `pera_sam_models.zip` (~10 MB) — extract `.h5` files into `model/server/assets/`.


In [ ]:
!pip install -q librosa tensorflow numpy scikit-learn pyyaml

In [ ]:
import os, glob

MACHINE_TYPES_TO_TRAIN = ['pump', 'slider', 'valve']
MACHINE_IDS = ['id_00', 'id_02', 'id_04', 'id_06']

# ── Auto-detect dataset root ────────────────────────────────────────────────
# Kaggle mounts datasets at /kaggle/input/<dataset-slug>/
# We search for the MIMII folder structure automatically.
def find_dataset_root():
    candidates = [
        '/kaggle/input',           # Kaggle standard
        '/content/drive/MyDrive',  # Colab + Drive
        '/content',                # Colab local
    ]
    for base in candidates:
        # Look for any folder that contains 'fan' or 'pump' subfolders
        hits = glob.glob(f'{base}/**/pump', recursive=True) + \
               glob.glob(f'{base}/**/fan',  recursive=True)
        if hits:
            # Walk up to find the root that contains all machine types
            for hit in hits:
                root = os.path.dirname(os.path.dirname(hit))  # go up past noise level folder
                if os.path.isdir(root):
                    print(f'  Found MIMII data under: {root}')
                    return root
    return None

DATASET_ROOT = find_dataset_root()
if DATASET_ROOT:
    print(f'Dataset root: {DATASET_ROOT}')
else:
    print('WARNING: Could not auto-detect dataset. Run the next cell to mount manually.')


### If auto-detection failed above

Run this cell to list what Kaggle has mounted, then update `DATASET_ROOT` manually.


In [ ]:
# List all mounted datasets
import os
if os.path.exists('/kaggle/input'):
    for d in os.listdir('/kaggle/input'):
        print('/kaggle/input/' + d)
        # Show first 2 levels inside
        for sub in os.listdir(f'/kaggle/input/{d}')[:8]:
            print('  ', sub)

# MANUAL OVERRIDE — set this if auto-detect above failed
# Example: DATASET_ROOT = '/kaggle/input/mimii-dataset'
# DATASET_ROOT = '/kaggle/input/YOUR-DATASET-SLUG'


In [ ]:
import glob, os

def find_wav_files(mtype, machine_id, split='normal'):
    """Search for wav files anywhere under DATASET_ROOT for given machine type/id/split."""
    pattern = f'{DATASET_ROOT}/**/{mtype}/{machine_id}/{split}/*.wav'
    files = sorted(glob.glob(pattern, recursive=True))
    if not files:
        # Also try without the machine_id subfolder level (some Kaggle datasets flatten)
        pattern2 = f'{DATASET_ROOT}/**/{mtype}/**/{split}/*.wav'
        files = sorted(glob.glob(pattern2, recursive=True))
    return files

# Preview: show how many files are available for each machine type
for mtype in MACHINE_TYPES_TO_TRAIN:
    for mid in MACHINE_IDS:
        n_files = find_wav_files(mtype, mid, 'normal')
        a_files = find_wav_files(mtype, mid, 'abnormal')
        status = 'OK' if n_files else 'NOT FOUND'
        print(f'  [{mtype}/{mid}] normal={len(n_files)} abnormal={len(a_files)}  [{status}]')


In [ ]:
import sys, numpy as np, librosa

N_MELS, N_FRAMES, N_FFT, HOP_LENGTH, POWER = 64, 5, 1024, 512, 2.0

def wav_to_vectors(fp):
    try:
        y, sr = librosa.load(fp, sr=None, mono=False)
        if y.ndim > 1: y = y[0]
        mel = librosa.feature.melspectrogram(
            y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS, power=POWER)
        lm  = 20.0 / POWER * np.log10(mel + sys.float_info.epsilon)
        vsz = lm.shape[1] - N_FRAMES + 1
        if vsz < 1: return None
        v = np.zeros((vsz, N_MELS * N_FRAMES))
        for t in range(N_FRAMES): v[:, N_MELS*t:N_MELS*(t+1)] = lm[:, t:t+vsz].T
        return v
    except Exception as e:
        print(f'  WARN: {fp}: {e}')
        return None

def load_files(fl, max_files=None):
    if max_files: fl = fl[:max_files]
    vecs = [wav_to_vectors(f) for f in fl]
    vecs = [v for v in vecs if v is not None]
    return np.concatenate(vecs, 0) if vecs else np.empty((0, N_MELS * N_FRAMES))

print('Feature extraction ready.')


In [ ]:
import tensorflow as tf
from tensorflow import keras

def build_ae(dim=320):
    i = keras.Input(shape=(dim,))
    h = keras.layers.Dense(64, activation='relu')(i)
    h = keras.layers.Dense(64, activation='relu')(h)
    h = keras.layers.Dense(8,  activation='relu')(h)
    h = keras.layers.Dense(64, activation='relu')(h)
    h = keras.layers.Dense(64, activation='relu')(h)
    o = keras.layers.Dense(dim)(h)
    m = keras.Model(i, o)
    m.compile(optimizer='adam', loss='mse')
    return m

print('Autoencoder ready.')


In [ ]:
import yaml
from sklearn.metrics import roc_auc_score

OUT = '/kaggle/working/pera_sam_assets'
os.makedirs(OUT, exist_ok=True)
metrics_all = {}

for mtype in MACHINE_TYPES_TO_TRAIN:
    for mid in MACHINE_IDS:
        id_s = mid.replace('id_', '')
        train_files = find_wav_files(mtype, mid, 'normal')
        if not train_files:
            print(f'[{mtype}/{mid}] No normal files found, skipping.')
            continue
        print(f'\n[{mtype}/{mid}] Training on {len(train_files)} normal files...')

        X = load_files(train_files)
        if X.shape[0] == 0:
            print(f'  Feature extraction failed, skipping.')
            continue

        model = build_ae(X.shape[1])
        model.fit(X, X, epochs=50, batch_size=512,
                  shuffle=True, validation_split=0.1, verbose=0)

        # Calibrate threshold at 95th percentile of normal MSE
        recon = model.predict(X, verbose=0)
        mse   = np.mean(np.square(X - recon), axis=1)
        thr   = float(np.percentile(mse, 95))

        # AUC score using abnormal samples if available
        auc = 0.5
        abn_files = find_wav_files(mtype, mid, 'abnormal')
        if abn_files:
            Xa = load_files(abn_files, max_files=30)
            Xn = load_files(train_files, max_files=30)
            if Xa.shape[0] > 0 and Xn.shape[0] > 0:
                sn = np.mean(np.square(Xn - model.predict(Xn, verbose=0)), axis=1)
                sa = np.mean(np.square(Xa - model.predict(Xa, verbose=0)), axis=1)
                try:
                    y_true  = np.concatenate([np.zeros(len(sn)), np.ones(len(sa))])
                    y_score = np.concatenate([sn, sa])
                    auc = float(roc_auc_score(y_true, y_score))
                except Exception as e:
                    print(f'  AUC calc failed: {e}')

        key = f'{mtype}_id_{id_s}_dataset'
        save_path = f'{OUT}/model_{mtype}_id_{id_s}_dataset.h5'
        model.save(save_path)
        metrics_all[key] = {'AUC': auc, 'threshold': thr}
        print(f'  Saved -> {save_path}')
        print(f'  AUC={round(auc, 4)}  threshold={round(thr, 4)}')

with open(f'{OUT}/metrics.yaml', 'w') as f:
    yaml.dump(metrics_all, f)

print('\nAll training complete!')
print('Files saved:')
for fn in sorted(os.listdir(OUT)):
    sz = os.path.getsize(f'{OUT}/{fn}') / 1024
    print(f'  {fn}  ({sz:.1f} KB)')


## Download

After running the training cell above, run this cell to create `pera_sam_models.zip`.

Then find it in the **Output** panel on the right side of the Kaggle notebook screen, or at `/kaggle/working/pera_sam_models.zip`.


In [ ]:
import shutil

zip_base = '/kaggle/working/pera_sam_models'
shutil.make_archive(zip_base, 'zip', OUT)
zip_path = zip_base + '.zip'
size_mb  = os.path.getsize(zip_path) / 1024 / 1024
print(f'Created: {zip_path}  ({size_mb:.1f} MB)')
print('Find it in the Kaggle Output panel on the right.')

# In Colab, this auto-downloads:
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass  # In Kaggle: use the Output panel


## Next Steps

1. Download `pera_sam_models.zip` from the Kaggle **Output** panel.
2. Extract it on your local PC.
3. Copy all `.h5` files into `model/server/assets/`.
4. Merge `metrics.yaml` entries into the existing `model/server/assets/metrics.yaml`.
5. Restart the ML backend: `python main.py` (or `start_server.bat`).

The server auto-detects new models at startup — no code changes needed.

---

### Which Kaggle MIMII Dataset to Add?

When you click **+ Add Data** in Kaggle and search **MIMII**, look for:

| Dataset name | Slug to use |
|---|---|
| `MIMII Dataset` by daisukelab | `daisukelab/mimii-dataset` |
| `MIMII Pump Sound Dataset` | search `mimii pump` |
| `MIMII Sound Dataset` | search `mimii sound` |

If none of those exist, you can also upload the zip files from [Zenodo record 3384388](https://zenodo.org/record/3384388) as a Kaggle dataset yourself (one-time upload, never local).
